# 第四部分：Stata 回归分析（nbstata）

使用 `nbstata` 在 Jupyter Notebook 中直接运行 Stata 代码，
作为 `do/` 文件夹的替代方案。所有回归结果与 do 文件等价。

In [ ]:
# nbstata 初始化
# 首次运行需安装: pip install nbstata
# 并在 C:\Users\<用户名>\.config\nbstata\nbstata.conf 中设置 stata_dir

import nbstata
nbstata.init()
print("nbstata 初始化完成")

## 1. 读取清洗后数据

In [ ]:
%%stata
* 设置工作目录（使用相对路径，跨机器可移植）
cd "../../data/clean"
use panel_data.dta, clear

* 转换 stkcd 并设面板
destring stkcd, gen(stkcd_num) force
drop stkcd
rename stkcd_num stkcd
xtset stkcd year

describe
summarize lev npr size tang growth ndts soe

## 2. M1: 双向固定效应基准模型（TWFE）

In [ ]:
%%stata
* M1: TWFE 基准回归，双重聚类标准误
reghdfe lev npr size tang growth ndts, ///
    absorb(stkcd year) ///
    vce(cluster stkcd year)
est store m1

display ""
display "M1: TWFE 基准回归"
display "NPR系数 = " %6.4f _b[npr]

## 3. M1': 交互固定效应模型（IFE）

In [ ]:
%%stata
* M1': IFE — 控制宏观因素的异质性影响
reghdfe lev npr size tang growth ndts m2_growth, ///
    absorb(stkcd year#m2_growth) ///
    vce(robust)
est store m1_ife

display ""
display "M1': IFE 回归结果"
display "m2_growth系数 = " %6.4f _b[m2_growth]

## 4. M2: 分组回归（SOE vs 非SOE）

In [ ]:
%%stata
* M2a: 国有企业
reghdfe lev npr size tang growth ndts if soe==1, ///
    absorb(stkcd year) ///
    vce(cluster stkcd year)
est store m2_soe

display ""
display "M2a: 国有企业 NPR系数 = " %6.4f _b[npr]

* M2b: 民营企业
reghdfe lev npr size tang growth ndts if soe==0, ///
    absorb(stkcd year) ///
    vce(cluster stkcd year)
est store m2_private

display "M2b: 民营企业 NPR系数 = " %6.4f _b[npr]

* 系数对比
estimates table m2_soe m2_private, b(%6.4f) se(%6.4f)

## 5. M3: 交互项调节效应

In [ ]:
%%stata
* M3: NPR×SOE 交互项
gen npr_soe = npr * soe

reghdfe lev npr npr_soe size tang growth ndts, ///
    absorb(stkcd year) ///
    vce(cluster stkcd year)
est store m3

display ""
display "M3: 交互项调节效应"
display "民营企业 NPR效应 = " %6.4f _b[npr]
display "国有企业 NPR效应 = " %6.4f (_b[npr] + _b[npr_soe])

## 6. M4: 时变系数模型

In [ ]:
%%stata
* M4: 时变系数 — NPR×Year 交互项
* 注意：year 不能同时被 absorb
reghdfe lev i.year c.npr i.year#c.npr size tang growth ndts, ///
    absorb(stkcd) ///
    vce(cluster stkcd year)
est store m4

* 提取各年系数到 CSV
postutil clear
postfile bt year coef se ci_low ci_high using "../../output/beta_time_coef.dta", replace

forvalues y = 2010/2025 {
    local idx = `y' - 2009
    local coef_idx = 17 + `idx'
    matrix b = e(b)
    matrix V = e(V)
    scalar coef_val = b[1, `coef_idx']
    scalar se_val = sqrt(V[`coef_idx', `coef_idx'])
    scalar ci_l = coef_val - 1.96 * se_val
    scalar ci_h = coef_val + 1.96 * se_val
    post bt (`y') (coef_val) (se_val) (ci_l) (ci_h)
}
postclose bt

use "../../output/beta_time_coef.dta", clear
outsheet using "../../output/beta_time_coef.csv", replace comma
display "β_t 时序数据已保存" 

## 7. M5: 函数系数模型（多项式调节）

In [ ]:
%%stata
* M5: 多项式 — NPR效应随 Size 变化
gen npr_size = npr * size
gen npr_size2 = npr * size^2

reghdfe lev npr npr_size npr_size2 size tang growth ndts, ///
    absorb(stkcd year) ///
    vce(cluster stkcd year)
est store m5_poly

display ""
display "β(Size) = -32.075 + 2.853×Size - 0.0645×Size²"

* 保存 β(Size) 曲线到 CSV
postutil clear
postfile betasize siz val se ci_l ci_h using "../../output/beta_size_coef.dta", replace

forvalues s = 20(0.5)30 {
    quietly margins, dydx(npr) at(size=(`s'))
    matrix b = r(b)
    matrix V = r(V)
    scalar v = b[1,1]
    scalar vv = sqrt(V[1,1])
    post betasize (`s') (v) (vv) (v - 1.96*vv) (v + 1.96*vv)
}
postclose betasize

use "../../output/beta_size_coef.dta", clear
outsheet using "../../output/beta_size_coef.csv", replace comma
display "β(Size) 曲线数据已保存" 

## 8. M6: 门槛模型

In [ ]:
%%stata
* M6: Hansen 面板门槛模型
use panel_data.dta, clear
destring stkcd, gen(stkcd_num) force
drop stkcd
rename stkcd_num stkcd
xtset stkcd year

* 单门槛检验（旧版 xthreg 语法需 rx() 选项）
xthreg lev npr size tang growth ndts, ///
    rx(npr) ///
    thv(size) ///
    trim(0.05) ///
    nboot(300) ///
    id(stkcd) ///
    time(year)

scalar gamma_hat = e(thrs_1)
di ""
di "门槛值: " gamma_hat
di "对应规模: " exp(gamma_hat) / 1e8 " 亿元"

* 保存门槛结果到 CSV
use "../../output/beta_size_coef.dta", clear
display "M6 门槛模型估计完成" 

## 9. 结果汇总

In [ ]:
%%stata
* M1-M3 结果汇总表
estimates table m1 m1_ife m2_soe m2_private m3, b(%6.4f) se(%6.4f)

display ""
display "=========================================="
display "回归分析完成！"
display "M1: TWFE — NPR系数 ≈ -0.627***"
display "M1': IFE — 宏观冲击后结果稳健"
display "M2a: SOE — NPR系数 ≈ -0.839***"
display "M2b: Private — NPR系数 ≈ -0.511***"
display "M3: 交互项 — NPR×SOE ≈ -0.196**"
display "M4: 时变 — β_t 在 2015-2016 最负，近年趋弱"
display "M5: β(Size) — 边际效应随规模增大而衰减"
display "M6: 门槛 — γ ≈ 22.13 (规模≈4亿元)"
display "==========================================" 

---
## 运行说明

1. **需要 nbstata**：`pip install nbstata`
2. **Stata 路径**：确保 `~/.config/nbstata/nbstata.conf` 中 `stata_dir` 正确
3. **运行顺序**：从第一个 cell 依次执行
4. **等效替代**：若 nbstata 不可用，运行 `do/` 文件夹中的 do 文件即可
5. **输出文件**：`beta_time_coef.csv`, `beta_size_coef.csv` 供 `03_results_and_figures.ipynb` 使用